# LiBr-H₂O solution properties

Evaluates specific enthalpy, entropy and specific volume of the liquid/two-phase
LiBr-water solution over a grid of temperatures and LiBr mass fractions at one or
more pressures, using the same property path as `SolutionConnection`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from tespy.tools.fluid_properties.functions import h_mix_pT, s_mix_pT, v_mix_pT
from tespy.tools.fluid_properties.wrappers import CoolPropWrapper

In [ ]:
# Create wrappers once and reuse them across all fluid_data dicts.
_libr_w = CoolPropWrapper("LiBr", back_end="INCOMP")
_h2o_w  = CoolPropWrapper("H2O")


def make_fluid_data(xi: float) -> dict:
    """Return fluid_data dict for LiBr-H2O at LiBr mass fraction *xi*."""
    return {
        "LiBr": {"mass_fraction": xi,       "wrapper": _libr_w},
        "H2O":  {"mass_fraction": 1.0 - xi, "wrapper": _h2o_w},
    }

In [ ]:
# Grid – change any of these freely
T_vals  = np.linspace(283.15, 498.15, 120)   # K  (10 °C … 180 °C)
xi_vals = np.arange(0.05, 0.71, 0.05)        # LiBr mass fraction
p_vals  = np.geomspace(500.0, 1_000_000.0, 20)   # Pa – arbitrary number of values

MIXING_RULE = "libr_water"

In [ ]:
def compute_props(p, xi_vals, T_vals):
    """Return (h, s, v) arrays shaped (len(xi_vals), len(T_vals)) in SI units."""
    shape = (len(xi_vals), len(T_vals))
    h = np.full(shape, np.nan)
    s = np.full(shape, np.nan)
    v = np.full(shape, np.nan)
    for i, xi in enumerate(xi_vals):
        fd = make_fluid_data(xi)
        for j, T in enumerate(T_vals):
            try:
                h[i, j] = h_mix_pT(p, T, fd, MIXING_RULE)
                s[i, j] = s_mix_pT(p, T, fd, MIXING_RULE)
                v[i, j] = v_mix_pT(p, T, fd, MIXING_RULE)
            except Exception:
                pass
    return h, s, v


results = {p: compute_props(p, xi_vals, T_vals) for p in p_vals}

In [ ]:
# h, s, v vs T at the median pressure — one line per xi, coloured by xi
p_ref = p_vals[len(p_vals) // 2]
h_ref, s_ref, v_ref = results[p_ref]

T_C    = T_vals - 273.15
norm   = mcolors.Normalize(vmin=xi_vals.min(), vmax=xi_vals.max())
cmap_xi = plt.get_cmap("plasma")

fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
for i, xi in enumerate(xi_vals):
    c = cmap_xi(norm(xi))
    axes[0].plot(T_C, h_ref[i] / 1e3, color=c)
    axes[1].plot(T_C, s_ref[i] / 1e3, color=c)
    axes[2].plot(T_C, v_ref[i] * 1e3, color=c)

sm = plt.cm.ScalarMappable(cmap=cmap_xi, norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, ax=axes, location="right", shrink=0.8, pad=0.02)
cb.set_label("ξ  [—]")

axes[0].set(xlabel="T [°C]", ylabel="h [kJ kg⁻¹]",       title="Specific enthalpy")
axes[1].set(xlabel="T [°C]", ylabel="s [kJ kg⁻¹ K⁻¹]",  title="Specific entropy")
axes[2].set(xlabel="T [°C]", ylabel="v [L kg⁻¹]",        title="Specific volume")
for ax in axes:
    ax.grid(True, alpha=0.4)

fig.suptitle(f"LiBr-H₂O  —  p = {p_ref:.0f} Pa", fontsize=13)
plt.show()

In [ ]:
# h, s, v vs T at fixed xi — one line per pressure, coloured by pressure
xi_ref = xi_vals[len(xi_vals) // 2]
xi_idx = np.argmin(np.abs(xi_vals - xi_ref))

norm   = mcolors.LogNorm(vmin=p_vals.min(), vmax=p_vals.max())
cmap_p = plt.get_cmap("viridis")

fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
for p in p_vals:
    h_p, s_p, v_p = results[p]
    c = cmap_p(norm(p))
    axes[0].plot(T_C, h_p[xi_idx] / 1e3, color=c)
    axes[1].plot(T_C, s_p[xi_idx] / 1e3, color=c)
    axes[2].plot(T_C, v_p[xi_idx] * 1e3, color=c)

sm = plt.cm.ScalarMappable(cmap=cmap_p, norm=norm)
sm.set_array([])
cb = fig.colorbar(sm, ax=axes, location="right", shrink=0.8, pad=0.02)
cb.set_label("p [Pa]")

axes[0].set(xlabel="T [°C]", ylabel="h [kJ kg⁻¹]",       title="Specific enthalpy")
axes[1].set(xlabel="T [°C]", ylabel="s [kJ kg⁻¹ K⁻¹]",  title="Specific entropy")
axes[2].set(xlabel="T [°C]", ylabel="v [L kg⁻¹]",        title="Specific volume")
for ax in axes:
    ax.grid(True, alpha=0.4)

fig.suptitle(f"LiBr-H₂O  —  ξ = {xi_ref:.2f}  ({len(p_vals)} pressure levels)", fontsize=13)
plt.show()